# Phase 1 — ASP Landscape: アリエールジェル vs アタック抗菌EX

**Purpose:** Map the complete pricing landscape per size for both brands.

| Step | Description |
|------|-------------|
| 1-1 | Monthly ASP trend per size — アリエールジェル (all sizes) |
| 1-2 | Monthly ASP trend per size — アタック抗菌EX (all sizes) |
| 1-3 | Side-by-side ASP comparison table: pre-renewal vs. post-renewal |
| 1-4 | Per-dose ASP gap (P&G capacity data + アタック ASP proxy) |

**ASP = `pos_sales_amt / pos_unit_sales_qty`** (no shelf_price_amt, no promo flag)  
**Created:** 2026-02-19

---
## 0. Imports & Connection

In [20]:
import os
import pandas as pd
import numpy as np
import warnings
from dotenv import load_dotenv
import databricks.sql as sql
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

warnings.filterwarnings('ignore')
pd.set_option('display.max_rows', 200)
pd.set_option('display.float_format', lambda x: f'{x:,.1f}')

def _find_japanese_font():
    for name in ['MS Gothic', 'MS PGothic', 'Yu Gothic', 'Meiryo', 'IPAexGothic']:
        if name in {f.name for f in fm.fontManager.ttflist}:
            return name
    return None

_jp_font = _find_japanese_font()
if _jp_font:
    plt.rcParams['font.family'] = _jp_font
    print(f'✅ Japanese font: {_jp_font}')

load_dotenv(dotenv_path='../../.env')
DATABRICKS_HOST      = os.getenv('DATABRICKS_HOST')
DATABRICKS_TOKEN     = os.getenv('DATABRICKS_TOKEN')
DATABRICKS_HTTP_PATH = os.getenv('DATABRICKS_HTTP_PATH')
assert all([DATABRICKS_HOST, DATABRICKS_TOKEN, DATABRICKS_HTTP_PATH]), 'Missing .env credentials'
print('✅ Credentials loaded')

def execute_query(query: str) -> pd.DataFrame:
    with sql.connect(server_hostname=DATABRICKS_HOST, http_path=DATABRICKS_HTTP_PATH,
                     access_token=DATABRICKS_TOKEN) as conn:
        with conn.cursor() as cur:
            cur.execute(query)
            result = cur.fetchall()
            columns = [d[0] for d in cur.description]
            return pd.DataFrame(result, columns=columns)

✅ Japanese font: MS Gothic
✅ Credentials loaded


---
## 1. Parameters

In [21]:
ARIEL_GEL   = 'ｱﾘｴｰﾙｼﾞｪﾙ'
ATTACK_EX   = 'ｱﾀｯｸ抗菌EX'
SUB_CAT     = '洗濯洗剤'
CATEGORY    = 'Laundry'

ANALYSIS_START = '2025-01-01'
ANALYSIS_END   = '2026-01-31'
RENEWAL_MONTH  = '2025-05-01'

RETAILER_CODES = [
    'cds_8005', 'cds_8006', 'cds_8007', 'cds_8008', 'cds_8009',
    'cds_8010', 'cds_8011', 'cds_8012', 'cds_8013',
]
RETAILER_IN = ', '.join(f"'{c}'" for c in RETAILER_CODES)

# ── Size order (physical size: small → large) and exclusions ──────────
SIZE_ORDER     = ['本体通常', '詰替超特大', '詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ', '詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ', '詰替ﾒｶﾞｼﾞｬﾝﾎﾞ']
EXCLUDED_SIZES = ['ｿﾉﾀ', '詰替通常', '詰替超ｼﾞｬﾝﾎﾞ']

def order_and_filter_sizes(sizes_list):
    """Return sizes in SIZE_ORDER, excluding EXCLUDED_SIZES."""
    sizes_set = set(sizes_list) - set(EXCLUDED_SIZES)
    return [s for s in SIZE_ORDER if s in sizes_set]

# ── Size role mapping ─────────────────────────────────────────────────
SIZE_ROLE = {
    '本体通常':          'Trial Entry',
    '詰替超特大':        'Intermission',
    '詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ':  'Loyalty',
    '詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ': 'Loyalty',
    '詰替ﾒｶﾞｼﾞｬﾝﾎﾞ':    'Loyalty',
}

print(f'📋 Renewal breakpoint: {RENEWAL_MONTH}')
print(f'📋 Size order: {SIZE_ORDER}')
print(f'📋 Excluded sizes: {EXCLUDED_SIZES}')


📋 Renewal breakpoint: 2025-05-01
📋 Size order: ['本体通常', '詰替超特大', '詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ', '詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ', '詰替ﾒｶﾞｼﾞｬﾝﾎﾞ']
📋 Excluded sizes: ['ｿﾉﾀ', '詰替通常', '詰替超ｼﾞｬﾝﾎﾞ']


---
### 📏 Canonical Definitions (Cross-Notebook Standard)

| Term | Definition | Window |
|------|-----------|--------|
| **Trial Shopper** | A shopper who purchases the sub-brand/category with **no purchase history of that sub-brand/category in the prior 12 months** (365 days). This is a rolling lookback per purchase event, NOT first-ever. | 365-day lookback |
| **Repeat Shopper** | A trial shopper who makes at least one subsequent purchase of the **same sub-brand** within **180 days** (6 months) after their trial event. | 180-day forward window |
| **Lapsed Shopper** | A trial shopper who makes **no subsequent purchase** of the same sub-brand within **180 days** (6 months) after their trial event. | 180-day forward window |
| **ASP** | `SUM(pos_sales_amt) / SUM(pos_unit_sales_qty)` — weighted average selling price. | Per transaction/aggregation |
| **ASP Band (50 JPY bin)** | `FLOOR(ASP / 50) * 50` — ASP floored to nearest 50 JPY. Label: `ASP (50 JPY bin)`. | — |

**Repeat + Lapse are mutually exclusive and exhaustive** within the trial cohort.

> ⚠️ These definitions are enforced consistently across notebooks 00–05 in this analysis.

---
## 2. Step 1-1 & 1-2: Monthly ASP Trend per Size (Both Brands)

In [22]:
# ── Fetch monthly ASP data for all sizes, both brands ─────────────────
asp_trend_query = f"""
SELECT
    DATE_TRUNC('month', CAST(idpos.sales_period_group_end_date_part AS DATE)) AS month,
    prod.jp_sub_brand_alter_lang_name   AS sub_brand,
    prod.jp_segment_4_name              AS size_code,
    COUNT(DISTINCT idpos.shopper_key)   AS shoppers,
    SUM(idpos.pos_unit_sales_qty)       AS total_units,
    SUM(idpos.pos_sales_amt)            AS total_sales,
    SUM(idpos.pos_sales_amt) / SUM(idpos.pos_unit_sales_qty) AS weighted_asp
FROM cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw idpos
LEFT JOIN id_pos_ai_1.prod_dim_ext_vw prod
       ON idpos.prod_key = prod.prod_key
LEFT JOIN id_pos_ai_1.shopper_dim_generic_vw shopper
       ON idpos.shopper_key = shopper.shopper_key
WHERE idpos.sales_period_group_end_date_part BETWEEN '{ANALYSIS_START}' AND '{ANALYSIS_END}'
  AND idpos.data_provider_code_part IN ({RETAILER_IN})
  AND prod.jp_category_name = '{CATEGORY}'
  AND prod.jp_sub_category_alter_lang_name = '{SUB_CAT}'
  AND prod.jp_sub_brand_alter_lang_name IN ('{ARIEL_GEL}', '{ATTACK_EX}')
  AND shopper.member_ind = 'Y'
  AND idpos.pos_unit_sales_qty > 0
GROUP BY 1, 2, 3
ORDER BY 1, 2, 3
"""

print('⏳ Fetching monthly ASP trends (all sizes, both brands)...', flush=True)
df_asp = execute_query(asp_trend_query)
df_asp['month'] = pd.to_datetime(df_asp['month'])
for col in ['shoppers', 'total_units', 'total_sales', 'weighted_asp']:
    df_asp[col] = pd.to_numeric(df_asp[col])

print(f'✅ {len(df_asp)} rows fetched')
print(f'   Ariel sizes: {df_asp[df_asp["sub_brand"]==ARIEL_GEL]["size_code"].nunique()}')
print(f'   Attack sizes: {df_asp[df_asp["sub_brand"]==ATTACK_EX]["size_code"].nunique()}')

⏳ Fetching monthly ASP trends (all sizes, both brands)...
✅ 182 rows fetched
   Ariel sizes: 8
   Attack sizes: 7


In [23]:
# ── Ariel Gel: ASP trend per size ─────────────────────────────────────
ariel_data = df_asp[(df_asp['sub_brand'] == ARIEL_GEL) & (~df_asp['size_code'].isin(EXCLUDED_SIZES))].copy()
ariel_sizes = order_and_filter_sizes(ariel_data['size_code'].unique())

n_sizes = len(ariel_sizes)
n_cols = 2
n_rows = (n_sizes + 1) // 2

fig = make_subplots(rows=n_rows, cols=n_cols,
                    subplot_titles=[f'アリエールジェル {s}' for s in ariel_sizes],
                    vertical_spacing=0.08, horizontal_spacing=0.08)

renewal_date = pd.Timestamp(RENEWAL_MONTH)

# Ensure tz-consistency
if ariel_data['month'].dt.tz is not None:
    renewal_date = renewal_date.tz_localize(ariel_data['month'].dt.tz)

for idx, size in enumerate(ariel_sizes):
    row = idx // n_cols + 1
    col = idx % n_cols + 1
    subset = ariel_data[ariel_data['size_code'] == size].sort_values('month')

    # Color code: blue pre-renewal, red post-renewal
    pre = subset[subset['month'] < renewal_date]
    post = subset[subset['month'] >= renewal_date]

    if len(pre) > 0:
        fig.add_trace(go.Scatter(x=pre['month'], y=pre['weighted_asp'],
                                 mode='lines+markers', name=f'{size} (Pre)',
                                 line=dict(color='#6495ED'), showlegend=(idx==0)),
                      row=row, col=col)
    if len(post) > 0:
        fig.add_trace(go.Scatter(x=post['month'], y=post['weighted_asp'],
                                 mode='lines+markers', name=f'{size} (Post)',
                                 line=dict(color='#FF6347'), showlegend=(idx==0)),
                      row=row, col=col)

    fig.add_vline(x=RENEWAL_MONTH, line_dash='dash', line_color='gray',
                  opacity=0.5, row=row, col=col)

fig.update_layout(height=300*n_rows, title_text='アリエールジェル: Monthly ASP by Size (Pre vs Post Renewal)',
                  template='plotly_white')
fig.update_yaxes(title_text='ASP (JPY)')
fig.show()


In [24]:
# ── Attack 抗菌EX: ASP trend per size ────────────────────────────────
attack_data = df_asp[(df_asp['sub_brand'] == ATTACK_EX) & (~df_asp['size_code'].isin(EXCLUDED_SIZES))].copy()
attack_sizes_list = order_and_filter_sizes(attack_data['size_code'].unique())

if len(attack_sizes_list) > 0:
    n_s = len(attack_sizes_list)
    n_c = 2
    n_r = (n_s + 1) // 2

    fig2 = make_subplots(rows=n_r, cols=n_c,
                         subplot_titles=[f'アタック抗菌EX {s}' for s in attack_sizes_list],
                         vertical_spacing=0.08, horizontal_spacing=0.08)

    for idx, size in enumerate(attack_sizes_list):
        row = idx // n_c + 1
        col = idx % n_c + 1
        subset = attack_data[attack_data['size_code'] == size].sort_values('month')
        fig2.add_trace(go.Scatter(x=subset['month'], y=subset['weighted_asp'],
                                  mode='lines+markers', name=size,
                                  line=dict(color=px.colors.qualitative.Set2[idx % 8])),
                       row=row, col=col)

    fig2.update_layout(height=300*n_r, title_text='アタック抗菌EX: Monthly ASP by Size',
                       template='plotly_white')
    fig2.update_yaxes(title_text='ASP (JPY)')
    fig2.show()
else:
    print('⚠️ No Attack data found')


In [13]:
# ── Step 1-3 (NEW): Combined Ariel + Attack ASP monthly trend per size ─
# One subplot per size; both brands plotted on same axes for direct comparison
combined_data = df_asp[~df_asp['size_code'].isin(EXCLUDED_SIZES)].copy()
combined_sizes = order_and_filter_sizes(combined_data['size_code'].unique())

if len(combined_sizes) == 0:
    print('⚠️ No data for combined chart after exclusion')
else:
    renewal_ts = pd.Timestamp(RENEWAL_MONTH)
    if combined_data['month'].dt.tz is not None:
        renewal_ts = renewal_ts.tz_localize(combined_data['month'].dt.tz)

    n_s = len(combined_sizes)
    n_c = 2
    n_r = (n_s + n_c - 1) // n_c

    fig_comb = make_subplots(rows=n_r, cols=n_c,
                             subplot_titles=combined_sizes,
                             vertical_spacing=0.1, horizontal_spacing=0.1)

    BRAND_CFG = {
        ARIEL_GEL: dict(color='#1E90FF', dash='solid',  label='Ariel Gel'),
        ATTACK_EX: dict(color='#FF6347', dash='dot',    label='Attack 抗菌EX'),
    }

    for idx, size in enumerate(combined_sizes):
        r = idx // n_c + 1
        c = idx % n_c + 1
        for brand, cfg in BRAND_CFG.items():
            subset = combined_data[
                (combined_data['sub_brand'] == brand) & (combined_data['size_code'] == size)
            ].sort_values('month')
            if len(subset) == 0:
                continue
            fig_comb.add_trace(go.Scatter(
                x=subset['month'], y=subset['weighted_asp'],
                mode='lines+markers', name=cfg['label'],
                line=dict(color=cfg['color'], dash=cfg['dash']),
                marker=dict(size=5),
                showlegend=(idx == 0)
            ), row=r, col=c)
        fig_comb.add_vline(x=RENEWAL_MONTH, line_dash='dash', line_color='gray',
                           opacity=0.4, row=r, col=c)

    fig_comb.update_layout(
        height=320 * n_r,
        title_text='アリエールジェル vs アタック抗菌EX — Monthly ASP per Size',
        template='plotly_white',
        legend=dict(orientation='h', yanchor='bottom', y=-0.08)
    )
    fig_comb.update_yaxes(title_text='ASP (JPY)')
    fig_comb.show()


In [25]:

# ── Step 1-3 Extra: Unit Ratio vs ASP Gap — How does the price gap move units? ──
#
# Left Y  : Unit ratio = Ariel units / Attack units × 100 (%)
#           > 100  → Ariel selling MORE units than Attack that month
#           < 100  → Ariel selling FEWER units than Attack
#           Base of comparison is always Attack = denominator
#
# Right Y : ASP gap = Ariel ASP − Attack ASP (JPY, bar chart)
#           Orange → Ariel is more expensive than Attack (gap > 0)
#           Green  → Ariel is cheaper than Attack      (gap ≤ 0)
#
# Interpretation:
#   If the ratio drops *exactly* when the bars turn orange → price GAP is the driver
#   If the ratio was already falling before bars went orange → absolute price is the driver
#   If ratio rises when gap widens → Ariel value-for-size is resonating despite higher price

_ug = df_asp[~df_asp['size_code'].isin(EXCLUDED_SIZES)].copy()
_plot_sizes = order_and_filter_sizes(_ug['size_code'].unique())

# Monthly totals per brand per size
_units = (
    _ug.groupby(['month', 'sub_brand', 'size_code'])['total_units']
    .sum().reset_index()
)
_asp_m = (
    _ug.groupby(['month', 'sub_brand', 'size_code'])
    .apply(lambda d: d['total_sales'].sum() / d['total_units'].sum())
    .reset_index(name='weighted_asp')
)

n_s = len(_plot_sizes)
n_c = 2
n_r = (n_s + n_c - 1) // n_c

_specs = [[{"secondary_y": True}, {"secondary_y": True}] for _ in range(n_r)]

fig_ug = make_subplots(
    rows=n_r, cols=n_c,
    subplot_titles=_plot_sizes,
    specs=_specs,
    vertical_spacing=0.14,
    horizontal_spacing=0.12,
)

_renewal_ts = pd.Timestamp(RENEWAL_MONTH)
if _ug['month'].dt.tz is not None:
    _renewal_ts = _renewal_ts.tz_localize(_ug['month'].dt.tz)

for idx, size in enumerate(_plot_sizes):
    r = idx // n_c + 1
    c = idx % n_c + 1

    # ── Unit ratio: Ariel / Attack × 100 ─────────────────────────────
    _ariel_u = _units[(_units['sub_brand'] == ARIEL_GEL) & (_units['size_code'] == size)][['month', 'total_units']].rename(columns={'total_units': 'ariel_units'})
    _attack_u = _units[(_units['sub_brand'] == ATTACK_EX) & (_units['size_code'] == size)][['month', 'total_units']].rename(columns={'total_units': 'attack_units'})
    _ratio = pd.merge(_ariel_u, _attack_u, on='month', how='inner').sort_values('month')
    _ratio['unit_ratio'] = (_ratio['ariel_units'] / _ratio['attack_units'] * 100).round(1)

    if len(_ratio) > 0:
        # Color-code line: above 100 = blue (Ariel ahead), below 100 = red (Attack ahead)
        fig_ug.add_trace(
            go.Scatter(
                x=_ratio['month'], y=_ratio['unit_ratio'],
                mode='lines+markers',
                name='Ariel / Attack Unit Ratio (%)',
                line=dict(color='#1E90FF', width=2.5),
                marker=dict(
                    size=7,
                    color=['#1E90FF' if v >= 100 else '#FF6347' for v in _ratio['unit_ratio']],
                    line=dict(width=1, color='white'),
                ),
                showlegend=(idx == 0),
                legendgroup='ratio',
            ),
            row=r, col=c, secondary_y=False,
        )
        # Reference line at 100 (parity)
        fig_ug.add_hline(y=100, line_dash='dot', line_color='#888', opacity=0.7,
                         row=r, col=c)

    # ── ASP gap ───────────────────────────────────────────────────────
    _ariel_a = _asp_m[(_asp_m['sub_brand'] == ARIEL_GEL) & (_asp_m['size_code'] == size)][['month', 'weighted_asp']].rename(columns={'weighted_asp': 'ariel_asp'})
    _attack_a = _asp_m[(_asp_m['sub_brand'] == ATTACK_EX) & (_asp_m['size_code'] == size)][['month', 'weighted_asp']].rename(columns={'weighted_asp': 'attack_asp'})
    _gap = pd.merge(_ariel_a, _attack_a, on='month', how='inner').sort_values('month')
    _gap['asp_gap'] = (_gap['ariel_asp'] - _gap['attack_asp']).round(0)

    if len(_gap) > 0:
        _bar_colors = ['#FFB347' if v >= 0 else '#66CDAA' for v in _gap['asp_gap']]
        fig_ug.add_trace(
            go.Bar(
                x=_gap['month'], y=_gap['asp_gap'],
                name='ASP Gap (Ariel−Attack, ¥)',
                marker_color=_bar_colors,
                opacity=0.40,
                showlegend=(idx == 0),
                legendgroup='gap',
            ),
            row=r, col=c, secondary_y=True,
        )
        fig_ug.add_hline(y=0, line_dash='dot', line_color='lightgray', opacity=0.6,
                         row=r, col=c)

    # ── Renewal vertical line ─────────────────────────────────────────
    fig_ug.add_vline(x=RENEWAL_MONTH, line_dash='dash', line_color='gray',
                     opacity=0.4, row=r, col=c)

    # ── Axis titles ───────────────────────────────────────────────────
    fig_ug.update_yaxes(title_text='Ariel / Attack Unit Ratio (%)', secondary_y=False,
                        row=r, col=c)
    fig_ug.update_yaxes(title_text='ASP Gap (¥)', secondary_y=True,
                        row=r, col=c, zeroline=True, zerolinecolor='lightgray')

fig_ug.update_layout(
    height=380 * n_r,
    title_text='Ariel / Attack Unit Ratio vs ASP Gap — by Size',
    template='plotly_white',
    barmode='overlay',
    legend=dict(orientation='h', yanchor='bottom', y=-0.06, x=0),
)
fig_ug.show()

print()
print('📌 How to read this chart:')
print('   Blue/Red line = Ariel unit ÷ Attack unit × 100')
print('     Blue dot (≥100) → Ariel selling more units than Attack that month')
print('     Red dot  (<100) → Ariel selling fewer units than Attack')
print('   Dotted line at 100 = parity')
print('   Orange bar = ASP gap > 0 → Ariel MORE expensive than Attack')
print('   Green bar  = ASP gap ≤ 0 → Ariel CHEAPER than Attack')
print()
print('   KEY DIAGNOSTIC QUESTION:')
print('   ① Ratio drops at the same time bars go orange (post-May 2025)')
print('     → RELATIVE price gap is driving Ariel unit loss')
print('   ② Ratio was already falling before May 2025')
print('     → ABSOLUTE price level is the structural problem')
print('   ③ Ratio holds steady or rises even as gap widens')
print('     → Ariel loyal buyers are price-insensitive at this size tier')



📌 How to read this chart:
   Blue/Red line = Ariel unit ÷ Attack unit × 100
     Blue dot (≥100) → Ariel selling more units than Attack that month
     Red dot  (<100) → Ariel selling fewer units than Attack
   Dotted line at 100 = parity
   Orange bar = ASP gap > 0 → Ariel MORE expensive than Attack
   Green bar  = ASP gap ≤ 0 → Ariel CHEAPER than Attack

   KEY DIAGNOSTIC QUESTION:
   ① Ratio drops at the same time bars go orange (post-May 2025)
     → RELATIVE price gap is driving Ariel unit loss
   ② Ratio was already falling before May 2025
     → ABSOLUTE price level is the structural problem
   ③ Ratio holds steady or rises even as gap widens
     → Ariel loyal buyers are price-insensitive at this size tier


---
## 3. Step 1-3: Pre vs Post Renewal ASP Comparison Table

In [26]:
# ── Build comparison table ────────────────────────────────────────────
df_asp['period'] = df_asp['month'].apply(lambda x: 'Pre-Renewal' if x < renewal_date else 'Post-Renewal')

comparison = df_asp.groupby(['sub_brand', 'size_code', 'period']).agg(
    total_sales=('total_sales', 'sum'),
    total_units=('total_units', 'sum'),
    avg_shoppers_per_month=('shoppers', 'mean'),
).reset_index()

comparison['weighted_asp'] = comparison['total_sales'] / comparison['total_units']

# Pivot for side-by-side view
pivot = comparison.pivot_table(
    index=['sub_brand', 'size_code'],
    columns='period',
    values=['weighted_asp', 'avg_shoppers_per_month', 'total_units'],
    aggfunc='first'
).round(1)

# Calculate ASP change
flat = comparison.pivot_table(
    index=['sub_brand', 'size_code'],
    columns='period',
    values='weighted_asp',
    aggfunc='first'
).reset_index()

if 'Pre-Renewal' in flat.columns and 'Post-Renewal' in flat.columns:
    flat['asp_change_%'] = ((flat['Post-Renewal'] - flat['Pre-Renewal']) / flat['Pre-Renewal'] * 100).round(1)
    flat['asp_change_jpy'] = (flat['Post-Renewal'] - flat['Pre-Renewal']).round(0)

print('=' * 80)
print('ASP Comparison: Pre-Renewal vs Post-Renewal')
print(f'Pre : {ANALYSIS_START} → {RENEWAL_MONTH}')
print(f'Post: {RENEWAL_MONTH} → {ANALYSIS_END}')
print('=' * 80)
print()

# ── Ariel summary ──────────────────────────────────────────────────────
print('▶ アリエールジェル')
ariel_comp = flat[flat['sub_brand'] == ARIEL_GEL].copy()
print(ariel_comp.to_string(index=False))

print()
print('▶ アタック抗菌EX')
attack_comp = flat[flat['sub_brand'] == ATTACK_EX].copy()
print(attack_comp.to_string(index=False))

ASP Comparison: Pre-Renewal vs Post-Renewal
Pre : 2025-01-01 → 2025-05-01
Post: 2025-05-01 → 2026-01-31

▶ アリエールジェル
sub_brand     size_code  Post-Renewal  Pre-Renewal  asp_change_%  asp_change_jpy
ｱﾘｴｰﾙｼﾞｪﾙ          本体通常         279.1        216.5          28.9            63.0
ｱﾘｴｰﾙｼﾞｪﾙ         詰替超特大         330.8        341.4          -3.1           -11.0
ｱﾘｴｰﾙｼﾞｪﾙ 詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ         868.6        823.5           5.5            45.0
ｱﾘｴｰﾙｼﾞｪﾙ     詰替超ｼﾞｬﾝﾎﾞ         748.8        642.6          16.5           106.0
ｱﾘｴｰﾙｼﾞｪﾙ          詰替通常         217.5        145.4          49.6            72.0
ｱﾘｴｰﾙｼﾞｪﾙ  詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ         761.2        614.4          23.9           147.0
ｱﾘｴｰﾙｼﾞｪﾙ   詰替ﾒｶﾞｼﾞｬﾝﾎﾞ         942.8      1,024.2          -7.9           -81.0
ｱﾘｴｰﾙｼﾞｪﾙ           ｿﾉﾀ       1,931.0      2,694.5         -28.3          -763.0

▶ アタック抗菌EX
sub_brand     size_code  Post-Renewal  Pre-Renewal  asp_change_%  asp_change_jpy
 ｱﾀｯｸ抗菌EX          本体通常         269.1        345.2         -22

In [16]:
# ── Visualize ASP comparison as grouped bar chart ─────────────────────
if 'Pre-Renewal' in flat.columns and 'Post-Renewal' in flat.columns:
    plot_data = flat.copy()
    plot_data['label'] = plot_data['sub_brand'] + ' | ' + plot_data['size_code']

    fig3 = go.Figure()
    fig3.add_trace(go.Bar(name='Pre-Renewal ASP', x=plot_data['label'],
                          y=plot_data['Pre-Renewal'], marker_color='#6495ED'))
    fig3.add_trace(go.Bar(name='Post-Renewal ASP', x=plot_data['label'],
                          y=plot_data['Post-Renewal'], marker_color='#FF6347'))

    fig3.update_layout(barmode='group', title='ASP Before vs After Renewal — All Sizes',
                       yaxis_title='Weighted ASP (JPY)', template='plotly_white',
                       height=500)
    fig3.show()
else:
    print('⚠️ Need both pre and post renewal data for comparison chart')

---
## 4. Step 1-4: Per-Dose ASP Gap (Ariel Capacity vs Attack Proxy)

In [17]:
# ── Fetch Ariel Gel capacity data from prod_dim (P&G only) ────────────
capacity_query = f"""
SELECT DISTINCT
    prod.jp_segment_4_name        AS size_code,
    prod.jp_prod_alter_lang_name  AS product_name,
    prod.jp_size_name             AS size_name,
    prod.jp_pack_size_name        AS pack_size_name
FROM id_pos_ai_1.prod_dim_ext_vw prod
WHERE prod.jp_sub_brand_alter_lang_name = '{ARIEL_GEL}'
  AND prod.jp_sub_category_alter_lang_name = '{SUB_CAT}'
ORDER BY 1, 2
"""

print('⏳ Fetching capacity data from product dimension...', flush=True)
df_cap = execute_query(capacity_query)
print(f'✅ {len(df_cap)} products found')
print()
print(df_cap.to_string(index=False))

⏳ Fetching capacity data from product dimension...
✅ 432 products found

    size_code                                              product_name      size_name pack_size_name
         None                                             ｱﾘｴｰﾙｼﾞｪﾙ1.72             詰替           None
         None                                           ｱﾘｴｰﾙｼﾞｪﾙﾍﾔ1.72             詰替           None
         本体通常                                   ｱﾘｴｰﾙ ｲｵﾝﾊﾟﾜｰｼﾞｪﾙ 1.1kg             本体           None
         本体通常                                  ｱﾘｴｰﾙ ｲｵﾝﾊﾟﾜｰｼﾞｪﾙ 本体 1kg             本体           None
         本体通常                        ｱﾘｴｰﾙ ｲｵﾝﾊﾟﾜｰｼﾞｪﾙ 消臭成分ｱｯﾌﾟ 本体 850g             本体           None
         本体通常                              ｱﾘｴｰﾙ ｲｵﾝﾊﾟﾜｰｼﾞｪﾙ 部屋干ｼ用 900g             本体           None
         本体通常                 ｱﾘｴｰﾙ ｲｵﾝﾊﾟﾜｰｼﾞｪﾙ 部屋干ｼ用 ｵﾘﾝﾋﾟｯｸﾛｺﾞ付ｷ 900g             本体           None
         本体通常                                ｱﾘｴｰﾙ ｲｵﾝﾊﾟﾜｰｼﾞｪﾙ ｸｰﾙ 850g             本体           None
         

In [18]:
# ── Build per-dose ASP table ──────────────────────────────────────────
# Manual capacity mapping — update based on Phase 0 product name inspection
# These are typical Ariel Gel capacities in grams/mL
ARIEL_CAPACITY_G = {
    '本体通常':          690,     # Typical 690g bottle
    '詰替超特大':        850,     # ~850g refill (competitive entry)
    '詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ':  1260,    # ~1260g ultra jumbo refill
    '詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ': 1520,    # ~1520g super ultra jumbo refill
    # Add more from Phase 0 product name inspection
}

print('⚠️  NOTE: Update ARIEL_CAPACITY_G based on actual product names from Phase 0.')
print('   Current values are typical estimates. Verify against jp_prod_alter_lang_name.')
print()

# Get post-renewal ASP per size for Ariel
ariel_post = df_asp[(df_asp['sub_brand'] == ARIEL_GEL) & (df_asp['month'] >= renewal_date)].copy()
ariel_asp_by_size = ariel_post.groupby('size_code').agg(
    total_sales=('total_sales', 'sum'),
    total_units=('total_units', 'sum')
).reset_index()
ariel_asp_by_size['weighted_asp'] = ariel_asp_by_size['total_sales'] / ariel_asp_by_size['total_units']

# Calculate per-gram ASP for Ariel
ariel_asp_by_size['capacity_g'] = ariel_asp_by_size['size_code'].map(ARIEL_CAPACITY_G)
ariel_asp_by_size['asp_per_gram'] = (ariel_asp_by_size['weighted_asp'] / ariel_asp_by_size['capacity_g']).round(2)
ariel_asp_by_size['sub_brand'] = ARIEL_GEL

# Attack: use ASP as proxy (no capacity data)
attack_post = df_asp[(df_asp['sub_brand'] == ATTACK_EX) & (df_asp['month'] >= renewal_date)].copy()
attack_asp_by_size = attack_post.groupby('size_code').agg(
    total_sales=('total_sales', 'sum'),
    total_units=('total_units', 'sum')
).reset_index()
attack_asp_by_size['weighted_asp'] = attack_asp_by_size['total_sales'] / attack_asp_by_size['total_units']
attack_asp_by_size['capacity_g'] = np.nan  # Not available for competitor
attack_asp_by_size['asp_per_gram'] = np.nan
attack_asp_by_size['sub_brand'] = ATTACK_EX

# Combine
dose_table = pd.concat([ariel_asp_by_size, attack_asp_by_size], ignore_index=True)

print('=' * 80)
print('Per-Dose ASP Comparison (Post-Renewal)')
print('  Ariel: ASP / capacity(g) | Attack: ASP only (no capacity data)')
print('=' * 80)
print(dose_table[['sub_brand', 'size_code', 'weighted_asp', 'capacity_g', 'asp_per_gram']].to_string(index=False))

⚠️  NOTE: Update ARIEL_CAPACITY_G based on actual product names from Phase 0.
   Current values are typical estimates. Verify against jp_prod_alter_lang_name.

Per-Dose ASP Comparison (Post-Renewal)
  Ariel: ASP / capacity(g) | Attack: ASP only (no capacity data)
sub_brand     size_code  weighted_asp  capacity_g  asp_per_gram
ｱﾘｴｰﾙｼﾞｪﾙ          本体通常         279.1       690.0           0.4
ｱﾘｴｰﾙｼﾞｪﾙ         詰替超特大         330.8       850.0           0.4
ｱﾘｴｰﾙｼﾞｪﾙ 詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ         868.6     1,520.0           0.6
ｱﾘｴｰﾙｼﾞｪﾙ     詰替超ｼﾞｬﾝﾎﾞ         748.8         NaN           NaN
ｱﾘｴｰﾙｼﾞｪﾙ          詰替通常         217.5         NaN           NaN
ｱﾘｴｰﾙｼﾞｪﾙ  詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ         761.2     1,260.0           0.6
ｱﾘｴｰﾙｼﾞｪﾙ   詰替ﾒｶﾞｼﾞｬﾝﾎﾞ         942.8         NaN           NaN
ｱﾘｴｰﾙｼﾞｪﾙ           ｿﾉﾀ       1,931.0         NaN           NaN
 ｱﾀｯｸ抗菌EX          本体通常         269.1         NaN           NaN
 ｱﾀｯｸ抗菌EX         詰替超特大         363.1         NaN           NaN
 ｱﾀｯｸ抗菌EX 詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ         

In [19]:
# ── Export Phase 1 results ────────────────────────────────────────────
output_file = 'phase1_asp_landscape.xlsx'

# Strip timezone for Excel compatibility
df_asp_export = df_asp.copy()
for col in df_asp_export.select_dtypes(include=['datetimetz']).columns:
    df_asp_export[col] = df_asp_export[col].dt.tz_localize(None)

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    df_asp_export.to_excel(writer, sheet_name='Monthly_ASP_Trend', index=False)
    flat.to_excel(writer, sheet_name='Pre_vs_Post_Comparison', index=False)
    dose_table.to_excel(writer, sheet_name='Per_Dose_ASP', index=False)

print(f'✅ Phase 1 results exported to {output_file}')

✅ Phase 1 results exported to phase1_asp_landscape.xlsx
